# CellIQ tissue model v2 (lung slides, LC25000)

Honest version of the tissue model. Changes from v1:
1. Near-duplicate (augmented) copies of the same photo are grouped, so they never sit in both train and test.
2. Stronger colour augmentation (random grayscale, saturation and hue jitter) so the model depends less on stain colour.
3. Extra checks: grayscale test, cancer-vs-normal risk score, threshold chosen on validation, Grad-CAM examples.

Research and educational prototype. Not for real diagnosis.

In [ ]:
DATASET = "lung_slides"
IMG_SIZE = 224
BATCH_SIZE = 32
HEAD_EPOCHS = 3
FINETUNE_EPOCHS = 4
SEED = 42
OUT_DIR = "/content/cellIQ_lung_slides_v2"

In [ ]:
import os, json, random, re, shutil
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Running on:", device)
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import kagglehub
path = kagglehub.dataset_download("javaidahmadwani/lc25000")
DATA_DIR = None
for root, dirs, _ in os.walk(path):
    if {"lung_aca", "lung_n", "lung_scc"} <= set(dirs):
        DATA_DIR = root
        break
print("DATA_DIR =", DATA_DIR)
assert DATA_DIR, "lung folders not found"

In [ ]:
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")
class_names = sorted([d for d in os.listdir(DATA_DIR)
                      if os.path.isdir(os.path.join(DATA_DIR, d)) and d.startswith("lung_")])
print("Groups:", class_names)
natural = lambda s: [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)]
files, labels = [], []
for idx, d in enumerate(class_names):
    folder = os.path.join(DATA_DIR, d)
    for f in sorted(os.listdir(folder), key=natural):
        if f.lower().endswith(IMG_EXT):
            files.append(os.path.join(folder, f)); labels.append(idx)
files = np.array(files); labels = np.array(labels)
print("Total images:", len(files), "| per group:", np.bincount(labels))

## Group near-duplicates, then split by group
LC25000 is about 750 original photos augmented into thousands of images. We embed every image with a pretrained network,
link images that look almost identical, and keep each linked group inside one split.

In [ ]:
mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
emb_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(mean, std)])

class PlainDS(Dataset):
    def __len__(self): return len(files)
    def __getitem__(self, i): return emb_tf(Image.open(files[i]).convert("RGB")), 0

emb = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT).to(device).eval()
emb.classifier = nn.Identity()
feats = []
with torch.no_grad():
    for x, _ in DataLoader(PlainDS(), batch_size=64, num_workers=2):
        feats.append(emb(x.to(device)).float().cpu())
feats = torch.cat(feats)
feats = F.normalize(feats - feats.mean(0), dim=1).to(device)
N = len(feats)

FLOOR = 0.75
pi, pj, ps = [], [], []
for i in range(0, N, 1024):
    s = feats[i:i+1024] @ feats.T
    r, c = torch.where(s >= FLOOR)
    pi.append((r + i).cpu().numpy()); pj.append(c.cpu().numpy()); ps.append(s[r, c].cpu().numpy())
pi, pj, ps = np.concatenate(pi), np.concatenate(pj), np.concatenate(ps)
print("Linked pairs at similarity >=", FLOOR, ":", len(pi))

def components(thr):
    m = ps >= thr
    G = coo_matrix((np.ones(m.sum()), (pi[m], pj[m])), shape=(N, N))
    n, gid = connected_components(G, directed=False)
    return n, gid

table = {}
for thr in (0.80, 0.825, 0.85, 0.875, 0.90, 0.925, 0.95):
    n, gid = components(thr)
    sizes = np.bincount(gid)
    table[thr] = (n, gid, int(sizes.max()))
    print(f"similarity >= {thr:.3f}: groups={n:5d} | biggest group={sizes.max():5d} | median size={int(np.median(sizes))}")

# Linking images by similarity can chain unrelated photos into one giant group (which breaks the split).
# Use the LOWEST threshold (most careful grouping) whose biggest group is still small enough to split fairly.
MAX_GROUP = 700
ok_thr = [t for t in table if table[t][2] <= MAX_GROUP]
SIM = min(ok_thr) if ok_thr else max(table)
n_groups, group_id, _ = table[SIM]
print("Chosen similarity threshold:", SIM, "->", n_groups, "groups")

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
sgkf = StratifiedGroupKFold(n_splits=7, shuffle=True, random_state=SEED)
folds = [idx for _, idx in sgkf.split(files, labels, group_id)]
te, va = folds[0], folds[1]
tr = np.concatenate(folds[2:])
for nm, ix in (("train", tr), ("val", va), ("test", te)):
    print(f"{nm:5s} n={len(ix):5d} per group: {np.bincount(labels[ix], minlength=len(class_names))}")

# leakage check: how many test images still have a similar-looking twin in train? (random split vs group split)
def twin_rates(test_idx, train_idx, thrs=(0.95, 0.90, 0.85, 0.80)):
    best = []
    for i in range(0, len(test_idx), 512):
        s = feats[test_idx[i:i+512]] @ feats[train_idx].T
        best.append(s.max(1).values.cpu().numpy())
    best = np.concatenate(best)
    return {str(t): round(float((best >= t).mean()), 4) for t in thrs}

rng = np.random.RandomState(SEED); perm = rng.permutation(N)
r_te, r_tr = perm[:len(te)], perm[len(te) + len(va):]
leak_random = twin_rates(r_te, r_tr)
leak_group = twin_rates(te, tr)
print("Share of test images with a near-identical twin in train, at similarity >= 0.95 / 0.90 / 0.85 / 0.80:")
print("  random split:", leak_random)
print("  group split :", leak_group, "(zero at or above the grouping threshold by construction; lower levels show what remains)")

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(), transforms.RandomRotation(20),
    transforms.RandomGrayscale(p=0.3),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.5, hue=0.1),
    transforms.ToTensor(), transforms.Normalize(mean, std)])
eval_tf = transforms.Compose([transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(mean, std)])
gray_tf = transforms.Compose([transforms.Grayscale(3), transforms.Resize((IMG_SIZE, IMG_SIZE)), transforms.ToTensor(), transforms.Normalize(mean, std)])

class ScanDataset(Dataset):
    def __init__(self, idxs, tf): self.idxs, self.tf = idxs, tf
    def __len__(self): return len(self.idxs)
    def __getitem__(self, i):
        j = self.idxs[i]
        return self.tf(Image.open(files[j]).convert("RGB")), int(labels[j])

train_dl = DataLoader(ScanDataset(tr, train_tf), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_dl = DataLoader(ScanDataset(va, eval_tf), batch_size=BATCH_SIZE, num_workers=2)
test_dl = DataLoader(ScanDataset(te, eval_tf), batch_size=BATCH_SIZE, num_workers=2)
gray_dl = DataLoader(ScanDataset(te, gray_tf), batch_size=BATCH_SIZE, num_workers=2)

model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
model = model.to(device)
counts = np.bincount(labels[tr], minlength=len(class_names))
weights = torch.tensor(counts.sum() / (len(class_names) * counts), dtype=torch.float32).to(device)
loss_fn = nn.CrossEntropyLoss(weight=weights)
print("Class weights:", weights.cpu().numpy().round(2))

In [ ]:
def run_epoch(dl, train, opt=None):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    with torch.set_grad_enabled(train):
        for x, y in dl:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = loss_fn(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item() * len(y); total += len(y)
            correct += (out.argmax(1) == y).sum().item()
    return loss_sum / total, correct / total

best_val, best_state = 1e9, None
def fit(epochs, lr, tag):
    global best_val, best_state
    opt = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    for ep in range(epochs):
        tl, ta = run_epoch(train_dl, True, opt)
        vl, va_ = run_epoch(val_dl, False)
        print(f"{tag} epoch {ep+1}: train loss {tl:.3f} acc {ta:.3f} | val loss {vl:.3f} acc {va_:.3f}", flush=True)
        if vl < best_val:
            best_val = vl; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

for p in model.parameters(): p.requires_grad = False
for p in model.classifier.parameters(): p.requires_grad = True
fit(HEAD_EPOCHS, 1e-3, "Phase 1 (head only)")
for p in model.parameters(): p.requires_grad = True
fit(FINETUNE_EPOCHS, 1e-4, "Phase 2 (fine-tune)")
model.load_state_dict(best_state)
torch.save(best_state, os.path.join(OUT_DIR, "model.pt"))
print("Training finished and best model saved.")

In [ ]:
@torch.no_grad()
def get_probs(dl):
    model.eval(); P, Y = [], []
    for x, y in dl:
        P.append(F.softmax(model(x.to(device)), dim=1).cpu().numpy()); Y.append(y.numpy())
    return np.concatenate(P), np.concatenate(Y)

pv, yv = get_probs(val_dl)
pt, yt = get_probs(test_dl)
pg, _ = get_probs(gray_dl)
pred, pred_g = pt.argmax(1), pg.argmax(1)

print(classification_report(yt, pred, target_names=class_names, digits=3))
cm = confusion_matrix(yt, pred)
print("Confusion matrix (rows = truth, columns = AI guess):\n", cm)
acc, acc_gray = float((pred == yt).mean()), float((pred_g == yt).mean())
print(f"Test accuracy (group split): {acc:.3f} | with colour removed (grayscale): {acc_gray:.3f}")
print("Multi-class AUC (one-vs-rest):", round(float(roc_auc_score(yt, pt, multi_class="ovr")), 3))

# cancer-vs-normal risk score = 1 - P(lung_n)
n_idx = class_names.index("lung_n")
risk_v, risk_t = 1 - pv[:, n_idx], 1 - pt[:, n_idx]
yb_v, yb_t = (yv != n_idx).astype(int), (yt != n_idx).astype(int)
cands = np.linspace(0.01, 0.99, 99)
ok = [t for t in cands if (risk_v[yb_v == 1] >= t).mean() >= 0.95]
thr = float(max(ok)) if ok else 0.5
pb = (risk_t >= thr).astype(int)
tn, fp, fn, tp = confusion_matrix(yb_t, pb, labels=[0, 1]).ravel()
bin_res = dict(threshold=round(thr, 3), sensitivity=round(float(tp / max(tp + fn, 1)), 3),
               specificity=round(float(tn / max(tn + fp, 1)), 3), auc=round(float(roc_auc_score(yb_t, risk_t)), 3))
print("Cancer vs normal (risk = 1 - P(lung_n)):", bin_res)

fig, ax = plt.subplots(figsize=(4.5, 4)); ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(class_names))); ax.set_yticks(range(len(class_names)))
ax.set_xticklabels(class_names, rotation=30, ha="right"); ax.set_yticklabels(class_names)
for i in range(len(cm)):
    for j in range(len(cm)): ax.text(j, i, cm[i, j], ha="center", va="center")
ax.set_xlabel("AI guess"); ax.set_ylabel("Truth"); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150); plt.show()

metrics = dict(test_accuracy=round(acc, 4), test_accuracy_grayscale=round(acc_gray, 4), n_test=int(len(te)),
               cancer_vs_normal=bin_res, similarity_threshold_for_groups=SIM, n_groups=int(n_groups),
               leak_rate_random_split=leak_random, leak_rate_group_split=leak_group,
               confusion_matrix=cm.tolist(), classes=class_names,
               note="Group-aware split (near-duplicate copies kept together). Still one public dataset: not clinically validated.")
json.dump(metrics, open(os.path.join(OUT_DIR, "metrics.json"), "w"), indent=2)
json.dump({"classes": class_names, "img_size": IMG_SIZE, "dataset": DATASET, "risk": "1 - P(lung_n)", "threshold": thr,
           "normalize_mean": mean, "normalize_std": std, "preprocess": "resize to 224x224, ImageNet normalise"},
          open(os.path.join(OUT_DIR, "meta.json"), "w"), indent=2)
print("metrics.json and meta.json saved")

In [ ]:
acts = {}
def fwd_hook(m, i, o):
    acts["a"] = o
    o.register_hook(lambda g: acts.__setitem__("g", g))
model.features[-1].register_forward_hook(fwd_hook)

def gradcam(path):
    img = Image.open(path).convert("RGB")
    x = eval_tf(img).unsqueeze(0).to(device)
    model.eval(); model.zero_grad()
    out = model(x); cls = out.argmax(1).item()
    out[0, cls].backward()
    w = acts["g"].mean(dim=(2, 3), keepdim=True)
    cam = F.relu((w * acts["a"]).sum(1, keepdim=True))
    cam = F.interpolate(cam, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)[0, 0]
    cam = ((cam - cam.min()) / (cam.max() - cam.min() + 1e-8)).detach().cpu().numpy()
    prob = F.softmax(out, dim=1)[0].detach().cpu().numpy()
    return img.resize((IMG_SIZE, IMG_SIZE)), cam, cls, prob

pick = random.Random(SEED).sample(list(te), 6)
fig, axes = plt.subplots(3, 4, figsize=(12, 9))
for k, j in enumerate(pick):
    img, cam, cls, prob = gradcam(files[j])
    ax1, ax2 = axes[k // 2][(k % 2) * 2], axes[k // 2][(k % 2) * 2 + 1]
    ax1.imshow(img); ax1.set_title("Truth: " + class_names[labels[j]], fontsize=9); ax1.axis("off")
    ax2.imshow(img); ax2.imshow(cam, cmap="jet", alpha=0.45)
    ax2.set_title(f"AI: {class_names[cls]} ({prob[cls]*100:.0f}%)", fontsize=9); ax2.axis("off")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "heatmap_examples.png"), dpi=130); plt.show()
wrong = int((pred != yt).sum())
print("Wrong test predictions:", wrong)

In [ ]:
shutil.make_archive("/content/cellIQ_lung_slides_v2", "zip", OUT_DIR)
print("Zip ready:", "/content/cellIQ_lung_slides_v2.zip", round(os.path.getsize("/content/cellIQ_lung_slides_v2.zip") / 1e6, 1), "MB")
print("\nSUMMARY")
print(json.dumps({k: v for k, v in metrics.items() if k != "confusion_matrix"}, indent=2))